# E2 Uncertainty Management

## Introduction

We study in this session the basics of uncertainty management and error propagation. En the [E1 notebook](E1_measurement_of_magnitudes.ipynb) 
we went over the basics of measurement of main magnitudes
on fluid flow from a fan. However, **every measurement is subject to an uncertainty**. When computing a global performance parameter (flow rate $Q$, 
pressure head $Y$ or efficiency $\eta$) this uncertainty propagates through the governing equations. It is important to consider this uncertainty in the 
final results in order to assess their reliability.


## Learning Objectives

- Learn the basics about measurement uncertainty according to the ISO 5168 standard and the Guide to the expression of uncertainty in measurement (GUM)
- Identify and quantify Type B uncertainties (sensor tolerance) for individual parameters using instrument specifications and ISO 5801 standard
- Master the use of the Python `uncertainties` package to automatically propagate measurement uncertainties
- Estimate uncertainty of final results of fan performance: flow rate, head, power, efficiency and dimensionless parameters

## Previous tasks (about 2 hours)

From [UNE-EN ISO 5801:2019](https://plataforma-aenormas-aenor-com.recursos.biblioteca.upc.edu/standard/UNE/N0061895) 
- Read Section 17 *Uncertainty analysis* (note that there is a recent modification of the standard in [UNE-EN ISO 5801:2019/A1:2025](https://plataforma-aenormas-aenor-com.recursos.biblioteca.upc.edu/standard/UNE/N0075114))

From [UNE-ISO 5168:2006](https://plataforma-aenormas-aenor-com.recursos.biblioteca.upc.edu/standard/UNE/N0036700)
- Read sections
  - 3 *Terms and definitions* Pay attention to the standardized vocabulary
  - 5 *General principles* 
  - 7 *Sources of uncertainty*
  - 8 *Sensitivity coefficients*
  - 9 *Combinations of uncertainties*
  - 10 *Expressions of results*

Section 4 of the ISO 5168 will be used as a reference for symbols.

## Some simple questions

## Tasks

Let's recover the example task for a centrifugal fan, studied in the [E1 Notebook](./E1_measurement_of_magnitudes.ipynb). These are the measured data:

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.float_format', '{:.4g}'.format) # To limit to 4 the number of significant digits in the output

P_e = np.array([1200, 1480, 1840, 2240, 2640, 3000, 3400, 3760]) # Power in watts
T = np.array([27.4, 25.7, 24.6, 24.0, 23.7, 23.6, 23.2, 22.5]) # Temperature in degrees Celsius
p = np.array([2.92, 3.06, 3.13, 3.13, 3.08, 2.92, 2.60, 2.23]) # Pressure in kPa, measured at the outlet of the system
p1 = np.array([2.92, 3.04, 3.09, 3.06, 2.96, 2.76, 2.38, 1.95]) # Pressure in kPa, measured at the inlet of the nozzle flow meter
Delta_p = np.array([0.0, 0.07, 0.27, 0.60, 1.05, 1.60, 2.38, 3.25]) # Pressure drop in kPa in nozzle flow meter (p2 - p1)
exp_data = pd.DataFrame({
    'P_e (W)': P_e,
    'T (°C)': T,
    'p (kPa)': p,
    'p1 (kPa)': p1,
    'Delta_p (kPa)': Delta_p
})
exp_data

,P_e (W),T (°C),p (kPa),p1 (kPa),Delta_p (kPa)
0,1200,27.4,2.92,2.92,0
1,1480,25.7,3.06,3.04,0.07
2,1840,24.6,3.13,3.09,0.27
3,2240,24,3.13,3.06,0.6
4,2640,23.7,3.08,2.96,1.05
5,3000,23.6,2.92,2.76,1.6
6,3400,23.2,2.6,2.38,2.38
7,3760,22.5,2.23,1.95,3.25


We are going to use the great python package [`uncertainties`](https://github.com/lmfit/uncertainties) that allows to make 
mathematical operations with quantities and its uncertainties, propagating the error. Let's we recompute the data frame for the 
acquired experimental data and its uncertainties, taken from the technical documentation of the measurement devices. 

This is a simple example:

```python
from uncertainties import ufloat, unumpy

a = ufloat(10.1, 0.1) # Example of a value with uncertainty
print(f"a = {a}")
b = ufloat(5.0, 0.2)  # Another value with uncertainty
print(f"b = {b}")
print(f"a + b = {c}") 
d = a * b
print(f"a * b = {d}") 
e = a / b
print(f"a / b = {e}") 
f = a*unumpy.log(b)
print(f"a * ln(b) = {f}") 
```
Result:

```
a = 10.10+/-0.10
b = 5.00+/-0.20
a + b = 15.10+/-0.22
a * b = 50.5+/-2.1
a / b = 2.02+/-0.08
a * ln(b) = 16.3+/-0.4
```

In [ ]:
import uncertainties.unumpy as unp
from uncertainties import ufloat

# We define the errors in the measurements

w_p = 0.01 # kPa, error in pressure measurements
w_T = 0.4 # °C, error in temperature measurements
w_P_e = 10 # W, error in power measurements

# We create arrays of ufloat objects for each variable, combining the nominal values and their uncertainties
P_e_u = unp.uarray(P_e, w_P_e)
T_u = unp.uarray(T, w_T)
p_u = unp.uarray(p, w_p)
p1_u = unp.uarray(p1, w_p)
Delta_p_u = unp.uarray(Delta_p, w_p)
exp_data = pd.DataFrame({
    'P_e (W)': P_e_u,
    'T (°C)': T_u,
    'p (kPa)': p_u,
    'p1 (kPa)': p1_u,
    'Delta_p (kPa)': Delta_p_u
})
exp_data


,P_e (W),T (°C),p (kPa),p1 (kPa),Delta_p (kPa)
0,1200+/-10,27.4+/-0.4,2.920+/-0.010,2.920+/-0.010,0.000+/-0.010
1,1480+/-10,25.7+/-0.4,3.060+/-0.010,3.040+/-0.010,0.070+/-0.010
2,1840+/-10,24.6+/-0.4,3.130+/-0.010,3.090+/-0.010,0.270+/-0.010
3,2240+/-10,24.0+/-0.4,3.130+/-0.010,3.060+/-0.010,0.600+/-0.010
4,2640+/-10,23.7+/-0.4,3.080+/-0.010,2.960+/-0.010,1.050+/-0.010
5,3000+/-10,23.6+/-0.4,2.920+/-0.010,2.760+/-0.010,1.600+/-0.010
6,3400+/-10,23.2+/-0.4,2.600+/-0.010,2.380+/-0.010,2.380+/-0.010
7,3760+/-10,22.5+/-0.4,2.230+/-0.010,1.950+/-0.010,3.250+/-0.010


1. Compute rotational velocity and torque exactly as in the E1 notebook. The error will propagate to the new magnitudes

2. Repeat the same as in E1 notebook. Remember that ambient magnitudes (pressure and temperature) are also subject to uncertainty.

3. Compute viscosity for the point 1 in the test bench.

4. The computation of the flow rate is tricky. It cannot be done directly with uncertainties numerical objects, since the functions in fluid.flow_meter module accept only numbers.  

5. Compute density of air at the outlet of the fan and use mass flow rate to calculate volumetric flow rate in the fan. We need it to get velocity
   and total pressure.

6. Compute average velocity of air at fan outlet with flow rate and duct section

7. Estimate mechanical energy rise $Y = \frac{\Delta p_0}{\rho}$ in $\text{J/kg}$, taking $\rho$ as the average between the inlet and outlet densities.

8. Estimate mechanical efficiency as 
$$
\eta = \frac{\Delta p_0 Q}{\omega M}
$$

9. Again, scale flow rate, mechanical energy and consumed mechanical power to the idling rotational velocity $N_0 = 2750 \,\text{rpm}$, considering similitude relationships 

10.  Plot performance curves $Y(Q)$, $P/Q)$ and $\eta(Q)$ with the error bars and uncertainty bands. Draw conclusions.

11.  Compute additional columns with relative uncertainty for $Q_N$, $Y_N$, $P_N$ and $\eta$. Plot and check if it fulfills the ISO 5801 requirements 

## Your project